# Õppetund 01 - Sissejuhatus tehisintellekti agentidesse

Tere tulemast kursuse **AI Agentide alustajatele** esimesse õppetundi!

**Tehisintellekti agent** on programm, mis kasutab suurt keelemudelit (LLM) oma mõtlemismootorina ja võib võtta *tegevusi* reaalses maailmas – kutsuda API-sid, teha päringuid andmebaasides või käivitada koodi – eesmärgi saavutamiseks kasutaja nimel.

Selles märkmes ehitad oma esimese agendi: **Reisibüroo agendi**, mis soovitab puhkuse sihtkohti. Selle käigus õpid, kuidas:

1. Ühenduda Microsoft Foundry Agendi teenusega, kasutades **Microsoft Agendi raamistiku**.
2. Anda agendile **tööriist** – tavaline Python funktsioon, mida ta saab kutsuda.
3. Käivitada agent ja uurida tema vastust.
4. Voogedastada agendi vastust tokeni haaval.


## Seadistamine

Enne selle sülearvutusraamatu käivitamist veenduge, et teil on:

1. **Microsoft Foundry projekt** koos juurutatud vestlusmudeliga (nt `gpt-4.1-mini`).
2. **Sisse logitud Azure CLI-ga** — käivitage terminalis käsk `az login`.
3. **Määratud vajalikud keskkonnamuutujad:**
   - `AZURE_AI_PROJECT_ENDPOINT` — teie Microsoft Foundry projekti lõpp-punkt.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — teie juurutatud mudeli nimi.

Allolev lahter installib vajalikud Python paketid.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Oma esimese agendi loomine

Agendil on vaja kahte asja:

- **Juhised**, mis ütlevad talle *kes ta on* ja *kuidas käituda* (süsteemi viip).
- **Tööriistad** — Python funktsioonid, mis on kaunistatud `@tool` ja mida agent saab kutsuda info hankimiseks või toimingute sooritamiseks.

Allpool määratleme lihtsa tööriista, mis tagastab populaarsete puhkuse sihtkohtade nimekirja. Agent kasutab seda tööriista, kui kasutaja küsib reisisoovitusi.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Voogedastuse vastused

Interaktiivsema kogemuse jaoks saate agendi vastuse **voogedastada**. Täisvastuse ootamise asemel annab agent tekstitükke välja kohe, kui need genereeritakse. See on eriti kasulik vestluspõhistes liidestes, kus soovite väljundit reaalajas kuvada.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Kokkuvõte

Selles õppetükis õppisite, kuidas:

- **Luu luua pakkuja**, mis ühendub Microsoft Foundry Agent Service’iga `FoundryChatClient`i kaudu.
- **Määratleda tööriist** kasutades `@tool` dekoratsiooni, et agent saaks teie Python funktsioone kutsuda.
- **Käivita agent** kasutaja sõnumiga ja prindi selle vastus.
- **Voogedasta vastuseid** reaalajas väljundi jaoks.

Järgmises õppetükis uurime agentuuriraamistikke põhjalikumalt ning õpime, kuidas anda agentidele võimsamaid tööriistu ja mitmeastmelist loogikat.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
